In [ ]:
# Cell 1 — Installation
!pip install -q transformers peft datasets rouge-score requests huggingface_hub
!pip install -q --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.5 MB/s eta 0:00:00


In [ ]:
# Cell 2 — Imports
import json
import time
import requests
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from rouge_score import rouge_scorer as rs
from huggingface_hub import login

login("hf_token")  

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [ ]:
# Cell 3 — Load test data
with open("/data/test.json") as f:
    test_data = json.load(f)

print(f"Test samples: {len(test_data)}")
print(f"\nSample:")
print(json.dumps(test_data[0], indent=2))

Test samples: 200

Sample:
{
  "instruction": "How are protists traditionally classified?",
  "response": "animal-like, plant-like, or fungi-like. Protists are traditionally classified as animal-like, plant-like, or fungi-like. You will read more about each of these types of protists in the next lesson."
}


In [4]:
# Cell 4 — Scoring functions
def rouge_l(prediction: str, ground_truth: str) -> float:
    scorer = rs.RougeScorer(['rougeL'], use_stemmer=True)
    score  = scorer.score(ground_truth, prediction)
    return round(score['rougeL'].fmeasure, 4)

def exact_match(prediction: str, ground_truth: str) -> int:
    import re
    pred  = re.sub(r'\s+', ' ', prediction.strip().lower())
    truth = re.sub(r'\s+', ' ', ground_truth.strip().lower())
    return int(pred == truth)

In [5]:
# Cell 5 — Base model inference
BASE_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

print("Loading base model...")
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
base_model     = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=torch.float16,
    device_map={"": 0}
)
base_model.eval()
print("Base model loaded.")

def run_base_model(question: str) -> dict:
    prompt  = f"<s>[INST] {question} [/INST]"
    inputs  = base_tokenizer(prompt, return_tensors="pt").to("cuda")
    start   = time.time()
    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True
        )
    latency = round((time.time() - start) * 1000)
    answer  = base_tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer  = answer.split("[/INST]")[-1].strip()
    return {"answer": answer, "latency_ms": latency}

Loading base model...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Base model loaded.


In [6]:
# Cell 6 — Fine-tuned LoRA model inference
LORA_ADAPTER_ID = "Shuvam-Maity/edututor-mistral-lora"

print("Loading LoRA model...")
lora_model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_ID)
lora_model.eval()
print("LoRA model loaded.")

def run_lora_model(question: str) -> dict:
    prompt  = f"<s>[INST] {question} [/INST]"
    inputs  = base_tokenizer(prompt, return_tensors="pt").to("cuda")
    start   = time.time()
    with torch.no_grad():
        outputs = lora_model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True
        )
    latency = round((time.time() - start) * 1000)
    answer  = base_tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer  = answer.split("[/INST]")[-1].strip()
    return {"answer": answer, "latency_ms": latency}

Loading LoRA model...


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/13.6M [00:00<?, ?B/s]

LoRA model loaded.


In [7]:
# Cell 7 — AWQ model inference via FastAPI with error handling
FASTAPI_URL = "https://shuvam-maity-edututor-api.hf.space/generate"

def run_awq_model(question: str) -> dict:
    try:
        start    = time.time()
        response = requests.post(
            FASTAPI_URL,
            json={"question": question, "max_tokens": 200},
            timeout=60
        )
        latency = round((time.time() - start) * 1000)
        data    = response.json()
        return {"answer": data.get("answer", ""), "latency_ms": latency}
    except Exception as e:
        print(f"\nAWQ API error: {e}")
        return {"answer": "", "latency_ms": 0}

In [8]:
# Cell 8 — Run evaluation on all 200 test samples
results = {
    "Base Mistral-7B"     : {"rouge_l": [], "exact_match": [], "latency_ms": []},
    "Fine-tuned LoRA"     : {"rouge_l": [], "exact_match": [], "latency_ms": []},
    "Fine-tuned AWQ 4-bit": {"rouge_l": [], "exact_match": [], "latency_ms": []},
}

for i, sample in enumerate(test_data):
    question     = sample["instruction"]
    ground_truth = sample["response"]

    print(f"Sample {i+1}/{len(test_data)}", end="\r")

    # Base model
    out = run_base_model(question)
    results["Base Mistral-7B"]["rouge_l"].append(rouge_l(out["answer"], ground_truth))
    results["Base Mistral-7B"]["exact_match"].append(exact_match(out["answer"], ground_truth))
    results["Base Mistral-7B"]["latency_ms"].append(out["latency_ms"])

    # LoRA model
    out = run_lora_model(question)
    results["Fine-tuned LoRA"]["rouge_l"].append(rouge_l(out["answer"], ground_truth))
    results["Fine-tuned LoRA"]["exact_match"].append(exact_match(out["answer"], ground_truth))
    results["Fine-tuned LoRA"]["latency_ms"].append(out["latency_ms"])

    # AWQ model via FastAPI
    out = run_awq_model(question)
    results["Fine-tuned AWQ 4-bit"]["rouge_l"].append(rouge_l(out["answer"], ground_truth))
    results["Fine-tuned AWQ 4-bit"]["exact_match"].append(exact_match(out["answer"], ground_truth))
    results["Fine-tuned AWQ 4-bit"]["latency_ms"].append(out["latency_ms"])

print("Evaluation complete.")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Evaluation complete.


In [9]:
# Cell 9 — Compute and print results
summary = {}
for name, metrics in results.items():
    n = len(metrics["rouge_l"])
    summary[name] = {
        "rouge_l"    : round(sum(metrics["rouge_l"]) / n, 4),
        "exact_match": round(sum(metrics["exact_match"]) / n * 100, 2),
        "avg_latency": round(sum(metrics["latency_ms"]) / n)
    }

# Save results
with open("eval_results.json", "w") as f:
    json.dump(summary, f, indent=2)

# Print table
print(f"\n{'Model':<25} | {'ROUGE-L':>8} | {'Exact Match':>12} | {'Avg Latency':>12}")
print("-" * 70)
for name, metrics in summary.items():
    print(f"{name:<25} | {metrics['rouge_l']:>8} | {metrics['exact_match']:>10}% | {metrics['avg_latency']:>10}ms")


Model                     |  ROUGE-L |  Exact Match |  Avg Latency
----------------------------------------------------------------------
Base Mistral-7B           |   0.3346 |        1.5% |       4125ms
Fine-tuned LoRA           |   0.3404 |        1.0% |       3807ms
Fine-tuned AWQ 4-bit      |   0.3249 |        0.5% |        705ms
